In [1]:
from inference_functions import load_bit_phoneme_model, evaluate_model, decode_outputs
from dataset import getDatasetLoaders
import numpy as np

/home/ubuntu/miniconda/envs/speech-bci/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data_file = '/data2/neural_data/ptDecoder_ctc_both_char_phoneme'
trainLoaders, testLoaders, loadedData = getDatasetLoaders(
        data_file, 8, None, 
        False
    )

In [3]:
device = 'cuda'
bit_phoneme_filepath = "/data2/models/time_masked_transfomer_characters_phonemes_80ms_seed_0/"
model, args = load_bit_phoneme_model(bit_phoneme_filepath)
model = model.to(device)

/home/ubuntu/miniconda/envs/speech-bci/lib/python3.9/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3190.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [4]:
outputs, cer, per_day_cer, cer2, per_day_cer2 = evaluate_model(model, loadedData, args, partition='train', device='cuda')


/home/ubuntu/transformers_with_dietcorp/src/neural_decoder/augmentations.py:170: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at ../aten/src/ATen/native/Convolution.cpp:895.)
  return self.conv(input, weight=self.weight, groups=self.groups, padding="same")


CER DAY 0: 0.186826
CER2 DAY 0: 0.162962
CER DAY 1: 0.161901
CER2 DAY 1: 0.133007
CER DAY 2: 0.074652
CER2 DAY 2: 0.050287
CER DAY 3: 0.091851
CER2 DAY 3: 0.060693
CER DAY 4: 0.031945
CER2 DAY 4: 0.017411
CER DAY 5: 0.034029
CER2 DAY 5: 0.018944
CER DAY 6: 0.032451
CER2 DAY 6: 0.016165
CER DAY 7: 0.035898
CER2 DAY 7: 0.019511
CER DAY 8: 0.038870
CER2 DAY 8: 0.020582
CER DAY 9: 0.046151
CER2 DAY 9: 0.026858
CER DAY 10: 0.033069
CER2 DAY 10: 0.017935
CER DAY 11: 0.040826
CER2 DAY 11: 0.021844
CER DAY 12: 0.045087
CER2 DAY 12: 0.025732
CER DAY 13: 0.032279
CER2 DAY 13: 0.016071
CER DAY 14: 0.028467
CER2 DAY 14: 0.015460
CER DAY 15: 0.028094
CER2 DAY 15: 0.015482
CER DAY 16: 0.026222
CER2 DAY 16: 0.013389
CER DAY 17: 0.028919
CER2 DAY 17: 0.013060
CER DAY 18: 0.031243
CER2 DAY 18: 0.017085
CER DAY 19: 0.030059
CER2 DAY 19: 0.017168
CER DAY 20: 0.030888
CER2 DAY 20: 0.014920
CER DAY 21: 0.028268
CER2 DAY 21: 0.015239
CER DAY 22: 0.033784
CER2 DAY 22: 0.014962
CER DAY 23: 0.036157
CER2 DAY 2

In [56]:
len(outputs['logits'])

8800

In [63]:
phoneme_decoded_strs, character_decoded_strs, true_seq_strs = decode_outputs(outputs, 8800)

In [64]:

def topk_labels(logits: np.ndarray, K: int, vocab: list) -> np.ndarray:
    """
    Args:
        logits: np.ndarray of shape (T, N) where
                N = 1 + len(vocab) (index 0 = CTC blank, rest follow vocab order)
        K: number of top tokens to return
        vocab: list of labels (phonemes or characters)

    Returns:
        np.ndarray of shape (T, K) with label strings
    """
    # Build id -> label mapping
    id2label = ["~"] + vocab

    # Get top-K indices per timestep
    topk_ids = np.argsort(logits, axis=1)[:, -K:][:, ::-1]

    # Map to labels
    topk_labels = np.vectorize(lambda i: id2label[i])(topk_ids)

    return topk_labels


In [65]:
# Phone definitions and mappings
PHONE_DEF = [
    'AA', 'AE', 'AH', 'AO', 'AW',
    'AY', 'B',  'CH', 'D', 'DH',
    'EH', 'ER', 'EY', 'F', 'G',
    'HH', 'IH', 'IY', 'JH', 'K',
    'L', 'M', 'N', 'NG', 'OW',
    'OY', 'P', 'R', 'S', 'SH',
    'T', 'TH', 'UH', 'UW', 'V',
    'W', 'Y', 'Z', 'ZH'
]
PHONE_DEF_SIL = PHONE_DEF + ["<>"]

CHAR_VOCAB = [
    "<>",          # space token
    "!", ",", ".", "?", "'",   # punctuation (incl. apostrophe)
] + [chr(i) for i in range(ord('a'), ord('z') + 1)]  # 'a'..'z'


character_outputs = []
for character_output in outputs['logits']:
    
    topk_chars = topk_labels(character_output, K=10, vocab=CHAR_VOCAB)
    character_outputs.append(topk_chars)
    
    
    
phoneme_outputs = []
for phoneme_output in outputs['logits2']:
    
    topk_phones = topk_labels(phoneme_output, K=10, vocab=PHONE_DEF_SIL)
    phoneme_outputs.append(topk_phones)

In [75]:
print(phoneme_outputs[2000])

[['~' '<>' 'S' 'B' 'D' 'K' 'T' 'DH' 'L' 'N']
 ['~' '<>' 'B' 'S' 'D' 'K' 'L' 'R' 'T' 'DH']
 ['~' '<>' 'B' 'D' 'S' 'K' 'R' 'L' 'T' 'Z']
 ['~' '<>' 'D' 'S' 'B' 'K' 'T' 'R' 'L' 'IY']
 ['~' '<>' 'D' 'S' 'IY' 'K' 'B' 'AH' 'Z' 'N']
 ['~' '<>' 'S' 'K' 'Z' 'D' 'N' 'IY' 'B' 'AH']
 ['~' 'K' 'S' 'Z' 'N' 'M' 'D' 'T' 'L' '<>']
 ['~' 'K' '<>' 'AY' 'HH' 'S' 'D' 'T' 'M' 'N']
 ['~' 'AY' '<>' 'HH' 'K' 'T' 'M' 'L' 'DH' 'N']
 ['AY' '~' '<>' 'IH' 'AH' 'IY' 'HH' 'EY' 'T' 'AE']
 ['AY' '~' 'IH' 'IY' 'AH' 'EY' 'AE' 'UW' 'EH' 'OW']
 ['~' 'AY' '<>' 'T' 'K' 'Z' 'V' 'IY' 'S' 'N']
 ['~' '<>' 'AY' 'K' 'T' 'Z' 'V' 'S' 'M' 'IY']
 ['<>' '~' 'K' 'AY' 'T' 'AH' 'S' 'HH' 'IY' 'V']
 ['<>' 'K' '~' 'HH' 'AE' 'S' 'D' 'AH' 'T' 'G']
 ['K' '~' '<>' 'HH' 'AE' 'D' 'G' 'T' 'P' 'S']
 ['K' 'AE' '~' 'D' 'HH' 'T' '<>' 'N' 'S' 'P']
 ['AE' '~' 'K' 'AH' 'EH' 'AA' 'EY' 'OW' 'AW' 'IH']
 ['AE' 'N' '~' 'AH' 'EH' 'OW' 'AA' 'K' 'EY' 'D']
 ['N' '~' 'AE' 'AH' 'T' 'NG' 'D' 'EH' 'Z' 'OW']
 ['N' 'T' 'AH' '~' 'AE' 'NG' 'D' 'IH' '<>' 'Z']
 ['N' 'T' '~' 

In [76]:
from typing import Any, Iterable, Sequence
def rows_to_text(arr: Any, elem_sep: str = " ", row_sep: str = "\n") -> str:
    """
    Convert a 2-D numpy array (or list of lists) to a single string.
    - Each element in a row is joined with `elem_sep`.
    - Each row is joined with `row_sep` (newline by default).
    """
    # support ndarray or list-of-lists
    rows: Iterable = arr.tolist() if isinstance(arr, np.ndarray) else arr
    lines = []
    for r in rows:
        if isinstance(r, np.ndarray):
            r = r.tolist()
        lines.append(elem_sep.join(map(str, r)))
    return row_sep.join(lines)

In [80]:
import json
from pathlib import Path
OUT_JSONL = "/data2/jsonl/train.jsonl"
USER_HDR = "<|start_header_id|>user<|end_header_id|>"
ASST_HDR = "<|start_header_id|>assistant<|end_header_id|>"
EOT      = "<|eot_id|>"   # include if your format expects it
prompt = (
    'You are helping decode speech from neural activity in a paralyzed patient. '
    'For each 80 ms time bin of neural activity, a model trained with CTC loss provides the top 10 most likely characters and phonemes, listed from most likely to least likely. '
    'The characters are presented first starting from the first timepoint, and then the phonemes are presented after.'
    'The phonemes are represented in the ARPABET format.'
    'The special token "~" denotes the CTC blank and should never be included in the response, and "<>" denotes a space and should be converted to a single space. '
    'Use both sources of evidence—characters and phonemes—to recover the intended English sentence, treating phonemes as pronunciation cues to resolve homophones and character-level misspellings. '
    'Produce fluent, standard spelling and grammar, adding simple punctuation (.,?! ) only when clearly warranted by the content. '
    'Output only the final sentence text, with no explanations or metadata.'
)

with Path(OUT_JSONL).open("w", encoding="utf-8") as fout:
    
    for idx in range(len(character_outputs)):
        
        topk_chars = rows_to_text(character_outputs[idx])
        topk_phonemes = rows_to_text(phoneme_outputs[idx])
        ground_truth = true_seq_strs[idx]
        
        msg = (
            f"{USER_HDR}\n\n"
            f"{prompt}\n\n"
            f"{topk_chars}\n\n"
            f"{topk_phonemes}\n\n"
            f"{ASST_HDR}\n\n"
            f"{ground_truth}{EOT}"
        )
        
        obj = {"text": msg}
        
        fout.write(json.dumps(obj, ensure_ascii=False) + "\n")

In [81]:
print(obj['text'])

<|start_header_id|>user<|end_header_id|>

You are helping decode speech from neural activity in paralyzed patients. For each 80 ms time bin of neural activity, a model trained with CTC loss provides the top 10 most likely characters and phonemes, listed from most likely to least likely. The characters are presented first starting from the first timepoint, and then the phonemes are presented after.The phonemes are represented in the ARPABET format.The special token "~" denotes the CTC blank, and "<>" denotes a space and should be converted to a single space. Use both sources of evidence—characters and phonemes—to recover the intended English sentence, treating phonemes as pronunciation cues to resolve homophones and character-level misspellings. Produce fluent, standard spelling and grammar, adding simple punctuation (.,?! ) only when clearly warranted by the content. Output only the final sentence text, with no explanations or metadata.

~ t h <> s e y w d l
~ t h s <> e y w d l
~ t <>